# 07 · Swiss roll: barrido de λ del regularizador del decoder (`jac`)

En el notebook 06, `jac` con continuación recuperó el rectángulo (ae_warm 6/6, vae_warm 5/6 en m = 64) pero con λ = 1 empeoró la reconstrucción ×4–7, por encima del costo admitido (≤ 2×). Aquí se barre λ para encontrar la frontera entre geometría y reconstrucción.

**Criterio de rectángulo v2.** Se fijó antes de correr y se calibró solo con los datos del notebook 05. Una réplica en una m es «rectángulo» si se cumplen las tres condiciones:
1. **desplegada**: kNN(latente → carta) ≤ 10 × piso (topología);
2. **localmente isométrica**: distorsión local ≤ 2 × la del oráculo DMAP de la misma réplica en m = M;
3. **forma global**: la lectura afín del latente sobre la carta tiene R²_s ≥ 0.90 **y** R²_h ≥ 0.80.

La condición 3 corrige el fallo que mostró `iso`: serpentinas con poca distorsión local pero que no son un rectángulo. Como referencia de la calibración (05, m = 64, medianas): oráculo DMAP R²_s = 1.00 y R²_h = 0.99; VAE con continuación sin regularizador R²_s = 0.72 y R²_h = 0.16 (el arco aplastado).

| | Pregunta | Diseño | Criterio |
|---|---|---|---|
| **L1** | ¿Hay un λ con rectángulo fiable y costo aceptable? | `ae_warm`, `vae_warm`, λ ∈ {0, 0.03, 0.1, 0.3, 1}, 6 réplicas (mismas semillas y datos que el 06) | λ* = el λ con mayor tasa de rectángulo en m = M entre los que cumplen MSE_x mediana ≤ 2 × la de λ = 0; con empate, el menor λ |
| **L2** | ¿λ* es mejor que no regularizar? | λ* frente a λ = 0, m = M | Fisher exacto unilateral p < 0.05 |

Solo se entrena desde cero el modelo de m = 2, que es el inicio de cada cadena (`scratch_first_only`). Si adjuntas como *Input* la salida del 06 (`results_swiss_latente_R1_sin_reg` y `..._R3_jac`), se reutilizan como λ = 0 y λ = 1, que tienen las mismas semillas y los mismos datos. Si no, se corren aquí.

Duración: ~3 h con la salida del 06 adjunta y ~4.5 h sin ella.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_07.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_07.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β y resultados previos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
FACTOR = 64
for prev in [Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')] + sorted(Path('/kaggle/input').rglob('beta_sweep_swiss/beta_selection.json')):
    if prev.exists():
        FACTOR = json.loads(prev.read_text())['selected_factor']
        break
print('factor beta =', FACTOR)

def previo(tag):
    # salida del notebook 06, si está en esta sesión o adjunta como Input
    for base in [Path('/kaggle/working')] + [p.parent for p in sorted(Path('/kaggle/input').rglob(f'results_swiss_latente_{tag}'))]:
        f = base / f'results_swiss_latente_{tag}'
        if (f / 'metrics.csv').exists():
            return f
    return None
PREV = {0.0: previo('R1_sin_reg'), 1.0: previo('R3_jac')}
print({k: str(v) if v else 'se correrá aquí' for k, v in PREV.items()})

## Plan y estimación de tiempo

El orden es la prioridad: primero los λ intermedios, que son los nuevos; λ = 0 y λ = 1 solo se corren si no hay salida del 06.

In [ ]:
import math
import protocol as P
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
BASE.update(width=60., height=10.)
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 64]
M = max(LADDER)
SEEDS = list(range(6))                      # las mismas que el 06
LAMBDAS = [0.1, 0.03, 0.3, 0.0, 1.0]        # orden = prioridad
comun = dict(beta_factor=FACTOR, seeds=SEEDS, modes=LADDER, models=['ae', 'vae'], m_protocols=['scratch', 'warm'],
             mode_order='spectral', oracle_decoder=False, reg_k=10, scratch_first_only=True)
PLAN, OUT = [], {}
for lam in LAMBDAS:
    if PREV.get(lam) is not None:
        OUT[lam] = PREV[lam]
        continue
    tag = f'jac_l{lam:g}'
    p = dict(tag=tag, **comun, out=f'/kaggle/working/results_swiss_lambda_{tag}')
    if lam > 0:
        p.update(reg='jac', reg_weight=lam)
    PLAN.append(p)
    OUT[lam] = Path(p['out'])
SECONDS_PER_MODEL = 80 * 1.4                # jac: 2 pasadas extra del decoder por lote
if PLAN:
    P.show_plan(PLAN, 'swiss', SECONDS_PER_MODEL, COMMON)
else:
    print('nada que correr')

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(PLAN, BASE, ROOT, common=COMMON)

## Análisis

Rectángulo v2 = desplegada ∧ distorsión ≤ 2 × oráculo(M) ∧ R²_s ≥ 0.90 ∧ R²_h ≥ 0.80. Costo = mediana de MSE_x / mediana de MSE_x con λ = 0 (mismo modelo y m).

In [ ]:
from scipy.stats import fisher_exact
import protocol as P
from latent_figure import chart_r2_table
R2S, R2H = .90, .80
MODELOS = ['ae_warm', 'vae_warm']

def evalua(folder):
    d = P.load(folder)
    if d is None:
        return None
    q = P.with_chain_start(d)
    q = q[q.model.isin(MODELOS)].copy()
    ref = d[(d.model == 'oracle') & (d.m == M)].set_index('seed').local_distortion
    r2 = chart_r2_table(folder, MODELOS)
    q = q.merge(r2, on=['seed', 'model', 'm'], how='left')
    q['iso_ok'] = q.local_distortion <= 2*q.seed.map(ref)
    q['forma_ok'] = (q.r2_s >= R2S) & (q.r2_h >= R2H)
    q['rect'] = q.unfolded.astype(bool) & q.iso_ok & q.forma_ok
    return q

Q = {lam: evalua(f) for lam, f in OUT.items()}
Q = {lam: q for lam, q in Q.items() if q is not None}
assert 0.0 in Q, 'Falta λ = 0 (referencia de costo).'
filas = []
for lam, q in sorted(Q.items()):
    for (model, m), g in q.groupby(['model', 'm']):
        base = Q[0.0][(Q[0.0].model == model) & (Q[0.0].m == m)].mse_x.median()
        k, n = int(g.rect.sum()), len(g)
        lo, hi = P.clopper_pearson(k, n)
        filas.append(dict(lam=lam, model=model, m=int(m), rectangulo=f'{k}/{n}', tasa=k/n, ic95_inf=lo, ic95_sup=hi,
                          desplegadas=int(g.unfolded.astype(bool).sum()), iso_ok=int(g.iso_ok.sum()), forma_ok=int(g.forma_ok.sum()),
                          distorsion=g.local_distortion.median(), r2_s=g.r2_s.median(), r2_h=g.r2_h.median(),
                          mse_x=g.mse_x.median(), costo=g.mse_x.median()/base))
T = pd.DataFrame(filas)
display(T[T.m == M].style.format({'tasa': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'distorsion': '{:.3g}',
                                  'r2_s': '{:.3f}', 'r2_h': '{:.3f}', 'mse_x': '{:.4g}', 'costo': '×{:.2f}'})
        .set_caption(f'm = {M}: rectángulo v2 y costo en reconstrucción'))

# ---- L1: elección de lambda (regla fijada antes de correr) ----
LSTAR = {}
for model in MODELOS:
    t = T[(T.model == model) & (T.m == M) & (T.lam > 0) & (T.costo <= 2)]
    if not len(t):
        print(f'L1 {model}: ningún λ > 0 cumple costo ≤ 2×'); continue
    best = t.sort_values(['tasa', 'lam'], ascending=[False, True]).iloc[0]
    LSTAR[model] = best.lam
    print(f"L1 {model}: λ* = {best.lam:g} -> rectángulo {best.rectangulo}, costo ×{best.costo:.2f}, "
          f"distorsión {best.distorsion:.3g}, R²_h {best.r2_h:.3f}")

# ---- L2: λ* frente a λ = 0 ----
for model, lam in LSTAR.items():
    a = Q[lam][(Q[lam].model == model) & (Q[lam].m == M)]
    b = Q[0.0][(Q[0.0].model == model) & (Q[0.0].m == M)]
    ka, kb = int(a.rect.sum()), int(b.rect.sum())
    p = fisher_exact([[ka, len(a)-ka], [kb, len(b)-kb]], alternative='greater')[1]
    print(f"L2 {model}: λ*={lam:g} {ka}/{len(a)} vs λ=0 {kb}/{len(b)}, Fisher p = {p:.3g} -> "
          f"{'CUMPLE' if p < .05 else 'no cumple'}")
T.to_csv('/kaggle/working/lambda_tablas.csv', index=False)

In [ ]:
# ---- Frontera geometría–reconstrucción y figuras de latente ----
from latent_figure import latent_grid
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
lams = sorted(Q)
for model, mk in (('ae_warm', '^'), ('vae_warm', 'o')):
    t = T[(T.model == model) & (T.m == M)].sort_values('lam')
    x = [l if l > 0 else 0.01 for l in t.lam]
    axes[0].errorbar(x, t.tasa, yerr=[t.tasa-t.ic95_inf, t.ic95_sup-t.tasa], marker=mk, capsize=3, label=model)
    axes[1].plot(x, t.costo, marker=mk, label=model)
    axes[2].plot(t.mse_x, t.distorsion, marker=mk, label=model)
    for _, r in t.iterrows():
        axes[2].annotate(f'λ={r.lam:g}', (r.mse_x, r.distorsion), fontsize=7, xytext=(3, 3), textcoords='offset points')
axes[1].axhline(2, color='k', ls=':', lw=.8, label='costo máximo admitido')
for a, (yl, tl) in zip(axes, (('fracción rectángulo (m = M)', 'Rectángulo v2 (IC 95 %)'),
                              ('MSE_x / MSE_x(λ = 0)', 'Costo en reconstrucción'),
                              ('distorsión local (mediana)', 'Frontera (cada punto = un λ)'))):
    a.set(ylabel=yl, title=tl)
    a.legend(fontsize=7)
for a in axes[:2]:
    a.set(xscale='log', xlabel='λ (λ = 0 dibujado en 0.01)')
axes[1].set_yscale('log')
axes[2].set(xscale='log', yscale='log', xlabel='test MSE$_x$ (mediana)')
fig.tight_layout()
fig.savefig('/kaggle/working/lambda_frontera.png', dpi=200); fig.savefig('/kaggle/working/lambda_frontera.pdf')
plt.show()

for lam in lams:
    f = OUT[lam]
    rows = [dict(label='DMAP (lectura afín)', folder=f, model='oracle'),
            dict(label='AE continuación', folder=f, model='ae_warm'),
            dict(label='VAE continuación', folder=f, model='vae_warm')]
    latent_grid(rows, LADDER, f'/kaggle/working/lambda_latente_{lam:g}',
                f'Swiss roll 60×10 · jac λ = {lam:g}: latente 2-D alineado afínmente a la carta (test)')
    plt.show()

## Qué descargar

`/kaggle/working/results_swiss_lambda_*`, `lambda_frontera.png/.pdf`, `lambda_latente_*.png/.pdf` y `lambda_tablas.csv`.